In [13]:
! pip install pandas requests python-dotenv openpyxl xlrd jupyterlab

In [14]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("EIA_API_KEY")
STAGING_DIR = "../data/staging/"

def fetch_eia_data(url, params, label=""):
    """Fetch EIA API dengan pagination otomatis."""
    all_records = []
    params["api_key"] = API_KEY
    params["offset"] = 0
    params["length"] = 5000
    
    while True:
        resp = requests.get(url, params=params)
        raw = resp.json()
        
        records = raw.get("response", {}).get("data", [])
        total   = raw.get("response", {}).get("total", 0)
        
        if not records:
            print(f"  ⚠ Tidak ada data untuk {label}")
            print("  Response:", str(raw)[:500])
            break
        
        all_records.extend(records)
        print(f"  {label}: {len(all_records)}/{total} records")
        
        if len(all_records) >= int(total):
            break
        
        params["offset"] += params["length"]
    
    return pd.DataFrame(all_records)

In [15]:
print("Mengambil Retail Sales...")

df_retail = fetch_eia_data(
    url="https://api.eia.gov/v2/electricity/retail-sales/data",
    params={
        "frequency": "monthly",
        "data[0]": "sales",
        "data[1]": "price",
        "data[2]": "revenue",
        "facets[stateid][]": "US",
        "facets[sectorid][]": ["RES", "COM", "IND", "ALL"],
        "start": "2022-01",
        "end": "2024-12",
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
    },
    label="retail-sales"
)

print("\nKolom:", df_retail.columns.tolist())
print("Shape:", df_retail.shape)
df_retail.head(5)

Mengambil Retail Sales...
  retail-sales: 144/144 records

Kolom: ['period', 'stateid', 'stateDescription', 'sectorid', 'sectorName', 'sales', 'price', 'revenue', 'sales-units', 'price-units', 'revenue-units']
Shape: (144, 11)


,period,stateid,stateDescription,sectorid,sectorName,sales,price,revenue,sales-units,price-units,revenue-units
0,2022-01,US,U.S. Total,ALL,all sectors,338656.04633,11.24,38053.17761,million kilowatt hours,cents per kilowatt-hour,million dollars
1,2022-01,US,U.S. Total,COM,commercial,113605.09055,11.26,12793.64264,million kilowatt hours,cents per kilowatt-hour,million dollars
2,2022-01,US,U.S. Total,IND,industrial,83982.00591,7.19,6037.27387,million kilowatt hours,cents per kilowatt-hour,million dollars
3,2022-01,US,U.S. Total,RES,residential,140504.06917,13.64,19162.6998,million kilowatt hours,cents per kilowatt-hour,million dollars
4,2022-02,US,U.S. Total,ALL,all sectors,305863.07052,11.42,34929.48889,million kilowatt hours,cents per kilowatt-hour,million dollars


In [16]:
print("Mengambil Net Generation...")

df_gen = fetch_eia_data(
    url="https://api.eia.gov/v2/electricity/electric-power-operational-data/data",
    params={
        "frequency": "monthly",
        "data[0]": "generation",
        "facets[location][]": "US",
        "facets[fueltypeid][]": ["COL", "NG", "NUC", "WND", "SUN", "HYC", "GEO", "OTH"],
        "start": "2022-01",
        "end": "2024-12",
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
    },
    label="generation"
)

print("\nKolom:", df_gen.columns.tolist())
print("Shape:", df_gen.shape)
df_gen.head(5)

Mengambil Net Generation...
  generation: 3324/3324 records

Kolom: ['period', 'location', 'stateDescription', 'sectorid', 'sectorDescription', 'fueltypeid', 'fuelTypeDescription', 'generation', 'generation-units']
Shape: (3324, 9)


,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units
0,2022-01,US,U.S. Total,1,Electric Utility,COL,"coal, excluding waste coal",63817.58413,thousand megawatthours
1,2022-01,US,U.S. Total,1,Electric Utility,GEO,geothermal,95.99719,thousand megawatthours
2,2022-01,US,U.S. Total,1,Electric Utility,HYC,conventional hydroelectric,22394.80501,thousand megawatthours
3,2022-01,US,U.S. Total,1,Electric Utility,NG,natural gas,66875.19976,thousand megawatthours
4,2022-01,US,U.S. Total,1,Electric Utility,NUC,nuclear,39294.817,thousand megawatthours


In [17]:
os.makedirs(STAGING_DIR, exist_ok=True)

df_retail.to_csv(f"{STAGING_DIR}eia_retail_2022_2024.csv", index=False)
df_gen.to_csv(f"{STAGING_DIR}eia_generation_2022_2024.csv", index=False)

print("✓ EIA retail   →", df_retail.shape)
print("✓ EIA generation →", df_gen.shape)

✓ EIA retail   → (144, 11)
✓ EIA generation → (3324, 9)


In [18]:
RAW_DIR = "../data/raw/"

TARGET_COUNTRIES = [
    "United States", "China", "Germany",
    "Japan", "India", "United Kingdom",
    "France", "Brazil"
]

# Mapping: nama file → nama kolom output
WB_FILES = {
    "CPI Price, % y-o-y, nominal, seas. adj..xlsx"          : "cpi_pct_yoy",
    "GDP at market prices, current US$, millions, seas. adj..xlsx"      : "gdp_current_usd_mn",
    "Industrial Production, constant 2010 US$, seas. adj..xlsx"    : "industrial_prod_usd",
    "Unemployment Rate, seas. adj..xlsx"                     : "unemployment_rate",
    "Exchange rate, new LCU per USD extended backward, period average.xlsx"         : "exchange_rate_lcu_usd",
}

def read_wb_monthly(filename, indicator_name):
    """
    Baca sheet 'monthly' dari file Excel World Bank.
    Struktur: row 0 = nama negara (header), row 1 = kosong, row 2+ = data.
    Kolom 0 = periode (tahun atau bulan), kolom 1+ = nilai per negara.
    """
    filepath = os.path.join(RAW_DIR, filename)
    
    # Cek sheet yang tersedia
    xl = pd.ExcelFile(filepath)
    sheet = "monthly" if "monthly" in xl.sheet_names else xl.sheet_names[0]
    print(f"\n📄 {filename[:45]}... | sheet: '{sheet}'")
    
    # Baca dengan header di baris 0
    df = pd.read_excel(filepath, sheet_name=sheet, header=0, index_col=0)
    
    # Hapus baris kosong pertama (baris 1 di file = baris 0 setelah set header)
    df = df.dropna(how="all")
    
    # Index sekarang adalah periode (tahun float atau string bulan)
    print(f"   Index sample  : {df.index[:5].tolist()}")
    print(f"   Kolom sample  : {df.columns[:5].tolist()}")
    print(f"   Shape sebelum filter: {df.shape}")
    
    # Filter hanya negara target yang tersedia
    available_countries = [c for c in TARGET_COUNTRIES if c in df.columns]
    print(f"   Negara ditemukan: {available_countries}")
    df = df[available_countries]
    
    # Filter periode 2022-2024
    # Index bisa berupa angka (2022.0) atau string ('2022M01')
    def is_in_range(idx):
        idx_str = str(idx)
        return any(str(y) in idx_str for y in ["2022", "2023", "2024"])
    
    df = df[df.index.map(is_in_range)]
    print(f"   Shape setelah filter periode: {df.shape}")
    
    if df.empty:
        print(f"   ⚠ Data kosong setelah filter — cek nama file atau format periode")
        return pd.DataFrame()
    
    # Reset index dan rename kolom periode
    df = df.reset_index()
    df = df.rename(columns={df.columns[0]: "period_raw"})
    
    # Melt ke format long
    df_long = df.melt(
        id_vars="period_raw",
        var_name="country",
        value_name=indicator_name
    )
    
    # Buang baris yang nilainya NaN
    df_long = df_long.dropna(subset=[indicator_name])
    
    print(f"   ✓ Final shape  : {df_long.shape}")
    return df_long

In [19]:
wb_results = {}

for filename, indicator in WB_FILES.items():
    try:
        df = read_wb_monthly(filename, indicator)
        wb_results[indicator] = df
    except FileNotFoundError:
        print(f"  ✗ File tidak ditemukan: {filename}")
        print(f"    → Cek nama file persis di folder data/raw/")
    except Exception as e:
        print(f"  ✗ Error pada {filename}: {e}")


📄 CPI Price, % y-o-y, nominal, seas. adj..xlsx... | sheet: 'monthly'
   Index sample  : ['1996M05', '1996M06', '1996M07', '1996M08', '1996M09']
   Kolom sample  : ['Albania', 'United Arab Emirates', 'Armenia', 'Austria', 'Azerbaijan']
   Shape sebelum filter: (359, 109)
   Negara ditemukan: ['United States', 'China', 'Germany', 'Japan', 'India', 'United Kingdom', 'France', 'Brazil']
   Shape setelah filter periode: (36, 8)
   ✓ Final shape  : (288, 3)

📄 GDP at market prices, current US$, millions, ... | sheet: 'annual'
   Index sample  : [1997.0, 1998.0, 1999.0, 2000.0, 2001.0]
   Kolom sample  : ['Albania', 'Advanced Economies', 'Argentina', 'Armenia', 'Australia']
   Shape sebelum filter: (29, 103)
   Negara ditemukan: ['United States', 'China', 'Germany', 'Japan', 'India', 'United Kingdom', 'France', 'Brazil']
   Shape setelah filter periode: (3, 8)
   ✓ Final shape  : (24, 3)

📄 Industrial Production, constant 2010 US$, sea... | sheet: 'monthly'
   Index sample  : ['1996M05', '19

In [20]:
import pandas as pd

def expand_annual_to_monthly(df_annual, value_col):
    """
    Expand data tahunan ke bulanan dengan forward-fill.
    Input: df dengan period_raw = [2022.0, 2023.0, 2024.0]
    Output: df dengan period_raw = ['2022M01', '2022M02', ..., '2024M12']
    """
    results = []
    
    for country in df_annual["country"].unique():
        df_country = df_annual[df_annual["country"] == country].copy()
        
        # Buat mapping tahun -> nilai
        year_map = {}
        for _, row in df_country.iterrows():
            year = int(float(row["period_raw"]))
            year_map[year] = row[value_col]
        
        # Generate semua bulan 2022-2024
        for year in [2022, 2023, 2024]:
            for month in range(1, 13):
                period = f"{year}M{month:02d}"
                value = year_map.get(year, None)
                results.append({
                    "period_raw": period,
                    "country": country,
                    value_col: value
                })
    
    return pd.DataFrame(results)


# Terapkan ke GDP
gdp_annual = wb_results["gdp_current_usd_mn"]
print("Sebelum expand:", gdp_annual.shape)
print(gdp_annual.head(5))

gdp_monthly = expand_annual_to_monthly(gdp_annual, "gdp_current_usd_mn")
print("\nSetelah expand:", gdp_monthly.shape)
print(gdp_monthly.head(10))

# Timpa di wb_results
wb_results["gdp_current_usd_mn"] = gdp_monthly

Sebelum expand: (24, 3)
   period_raw        country  gdp_current_usd_mn
0      2022.0  United States          26054600.0
1      2023.0  United States          27811500.0
2      2024.0  United States          29298100.0
3      2022.0          China          18362751.0
4      2023.0          China          18285554.0

Setelah expand: (288, 3)
  period_raw        country  gdp_current_usd_mn
0    2022M01  United States          26054600.0
1    2022M02  United States          26054600.0
2    2022M03  United States          26054600.0
3    2022M04  United States          26054600.0
4    2022M05  United States          26054600.0
5    2022M06  United States          26054600.0
6    2022M07  United States          26054600.0
7    2022M08  United States          26054600.0
8    2022M09  United States          26054600.0
9    2022M10  United States          26054600.0


In [21]:
print("=== Ringkasan semua dataset World Bank ===\n")

for indicator, df in wb_results.items():
    if df.empty:
        print(f"⚠  {indicator}: KOSONG")
        continue
    
    print(f"✓  {indicator}")
    print(f"   Shape   : {df.shape}")
    print(f"   Period  : {sorted(df['period_raw'].unique())[:3]} ... {sorted(df['period_raw'].unique())[-3:]}")
    print(f"   Negara  : {sorted(df['country'].unique())}")
    print(f"   Sample nilai: {df.iloc[0].to_dict()}")
    print()

=== Ringkasan semua dataset World Bank ===

✓  cpi_pct_yoy
   Shape   : (288, 3)
   Period  : ['2022M01', '2022M02', '2022M03'] ... ['2024M10', '2024M11', '2024M12']
   Negara  : ['Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'United Kingdom', 'United States']
   Sample nilai: {'period_raw': '2022M01', 'country': 'United States', 'cpi_pct_yoy': 7.558806}

✓  gdp_current_usd_mn
   Shape   : (288, 3)
   Period  : ['2022M01', '2022M02', '2022M03'] ... ['2024M10', '2024M11', '2024M12']
   Negara  : ['Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'United Kingdom', 'United States']
   Sample nilai: {'period_raw': '2022M01', 'country': 'United States', 'gdp_current_usd_mn': 26054600.0}

✓  industrial_prod_usd
   Shape   : (288, 3)
   Period  : ['2022M01', '2022M02', '2022M03'] ... ['2024M10', '2024M11', '2024M12']
   Negara  : ['Brazil', 'China', 'France', 'Germany', 'India', 'Japan', 'United Kingdom', 'United States']
   Sample nilai: {'period_raw': '2022M01', 'country

In [22]:
name_map = {
    "cpi_pct_yoy"          : "wb_cpi",
    "gdp_current_usd_mn"   : "wb_gdp",
    "industrial_prod_usd"  : "wb_industrial",
    "unemployment_rate"    : "wb_unemployment",
    "exchange_rate_lcu_usd": "wb_exchange_rate",
}

for indicator, df in wb_results.items():
    if df.empty:
        print(f"⚠ Skip {indicator} — kosong")
        continue
    out_name = name_map.get(indicator, indicator)
    out_path = f"{STAGING_DIR}{out_name}_2022_2024.csv"
    df.to_csv(out_path, index=False)
    print(f"✓ {out_path} — {df.shape[0]} rows")

✓ ../data/staging/wb_cpi_2022_2024.csv — 288 rows
✓ ../data/staging/wb_gdp_2022_2024.csv — 288 rows
✓ ../data/staging/wb_industrial_2022_2024.csv — 288 rows
✓ ../data/staging/wb_unemployment_2022_2024.csv — 216 rows
✓ ../data/staging/wb_exchange_rate_2022_2024.csv — 288 rows
